# 03 · GAN, part 1: train a class-conditional StyleGAN2-ADA

Phase 3 of the project. A GAN learns from the training images what a leaf of each class looks like. Part 2
(`03_gan_generate`, written once this one has trained) then makes new images for the small classes and checks them.
This notebook:

1. writes the **train split only** (12,698 photos of 256 × 256 pixels, each with its class) into the GAN's training
   file. The validation and test photos are not read at all,
2. downloads NVIDIA's official StyleGAN2-ADA code and changes the few lines that no longer work with today's Python
   and PyTorch (listed in section 2),
3. trains **one GAN for all 10 classes** (class-conditional: it is told which class to draw) until the configured time cutoff, and
   saves a snapshot of the networks every 100 kimg,
4. measures the **FID** of every snapshot (how far the generated images are from the real ones; lower is better) and
   draws sample images of every class.

**Words used below.** *Generator (G)*: the network that draws images. *Discriminator (D)*: the network that learns to
tell real photos from generated ones; G learns by trying to fool it. *kimg*: a thousand real images shown to D; the
length of GAN training is counted in kimg. *ADA* (adaptive discriminator augmentation): the images D sees are
augmented (flipped, rotated, recoloured, ...) with a probability *p* that rises automatically when D starts to memorise
the training images. This is what lets StyleGAN2-ADA learn from a few thousand images, and the shared GAN learn
Mosaic virus from its 261 photos. Only D's input is augmented, so the generated images do not show the augmentations.

**Sessions.** A Kaggle session stops after 12 hours, and the GAN needs several (probably 3–5). Each session continues
from the last snapshot of the one before:

- **Session 1:** set up as below and commit.
- **Session 2 and later:** **Add Input** → **Your Work** → **Notebooks** → this notebook → **Add** (it must be the
  newest version: the input panel shows the version number), then commit again. The notebook finds the newest
  snapshot by itself and prints `Continuing from ...`. Without that input it starts again from the beginning.
- **When to stop:** when the FID has not improved for about 500 kimg (the chart at the end), or when the time box of
  the plan is used up (2 weeks, about 60 GPU hours).

**How to run it on Kaggle**

- **Add Input** → search `plantvillage-dataset` (by *abdallahalidev*) → **Add**.
- **Add Input** → **Your Work** → **Notebooks** → your saved notebook 01 → **Add**. This gives `splits.csv`.
- **Settings**: Accelerator **GPU T4 x2** (the GAN trains on both GPUs). **Internet on**: it downloads the GAN code
  from GitHub and the network the FID uses.
- **Save Version → Save & Run All (Commit)**. Training stops at `TRAIN_HOURS` from notebook start; FID and saving need extra time. The browser can be
  closed. Then open the version's **Output** tab and go through the checklist at the end.
- Download `results/` (a few MB) into the project folder as `results-03/`. The snapshots in `gan/snapshots/` (about
  300 MB each) stay on Kaggle: the next session and part 2 read them from there.

In [ ]:
import os, re, sys, json, time, shutil, pickle, signal, hashlib, zipfile, subprocess, threading, importlib.util
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch

NOTEBOOK_START = time.time()

# The GAN: StyleGAN2-ADA with the settings of its paper's 256 x 256 experiments ("paper256"), except a batch of 32
# instead of 64, for Kaggle's two GPUs
RES = 256              # image size: PlantVillage photos are 256 x 256 already (the classifiers resize them to 224)
BATCH = 32             # images per training step, shared by the GPUs
BATCH_PER_GPU = 16     # images one GPU works on at a time
MAX_GPUS = 2           # set to 1 if training on two GPUs hangs or fails (same results, about half the speed)
GAMMA = 1.0            # strength of D's R1 regularisation (keeps D smooth around the real images)
CHANNEL_BASE = 16384   # network width (paper256: half of StyleGAN2's full width)
MAPPING_LAYERS = 8     # layers of G's mapping network
MBSTD_GROUP = 8        # D's minibatch standard deviation layer looks at groups of 8 images
LEARNING_RATE = 0.0025 # Adam, for G and D
EMA_KIMG = 20          # the generator used for images (G_ema) is a running average of G, half-life 20 kimg
MIRROR = True          # also train on left-right flipped photos: a flipped leaf shows the same disease
ADA_TARGET = 0.6       # ADA raises p while D is surer than this about the real images (the paper's value)

# Sessions
TRAIN_HOURS = 6.75     # session 5: 8 h 23 min GPU quota available; leave time for FID and saving
MAX_KIMG = 10000       # never train longer than this; the time box will most likely end training earlier
TICK_KIMG = 4          # a progress line every 4 kimg
SNAPSHOT_KIMG = 100    # a snapshot every 100 kimg, and one when training stops
FID_IMAGES = 10000     # generated images per FID measurement
SAMPLE_SEED = 2026     # fixed random numbers for the FID and the sample images: the same in every session

# NVIDIA's code, at a fixed version (the last change to the repository, April 2023)
STYLEGAN_REPO = 'https://github.com/NVlabs/stylegan3.git'
STYLEGAN_COMMIT = 'c233a919a6faee6e36a316ddd4eddababad1adf9'

# The final split of notebook 01 (12,698 / 2,728 / 2,734 images) as a fingerprint of every image's path and split
SPLIT_FINGERPRINT = '7cf95c207559517a'

# Kaggle folders (the environment variables are only for a test run on another computer)
INPUT_ROOT = Path(os.environ.get('INPUT_ROOT', '/kaggle/input'))
OUT_DIR = Path(os.environ.get('OUT_DIR', '/kaggle/working'))   # saved as the notebook's output
WORK_DIR = Path(os.environ.get('WORK_DIR', '/tmp/gan'))        # not saved: the GAN code and its training file
GAN_OUT, RESULTS_OUT = OUT_DIR / 'gan', OUT_DIR / 'results'
for folder in (WORK_DIR, GAN_OUT, RESULTS_OUT):
    folder.mkdir(parents=True, exist_ok=True)
PROGRESS_KIND = 'tomato-leaf stylegan2-ada'   # marks this notebook's progress.json among the inputs
assert SNAPSHOT_KIMG % TICK_KIMG == 0 and BATCH % BATCH_PER_GPU == 0

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    try:   # a GPU this PyTorch version does not support fails at its first calculation
        torch.nn.functional.conv2d(torch.ones(1, 1, 4, 4, device=DEVICE), torch.ones(1, 1, 3, 3, device=DEVICE))
    except RuntimeError as e:
        raise RuntimeError(f'The GPU {torch.cuda.get_device_name()} does not work with this PyTorch ({e}). '
                           'Choose Settings → Accelerator → GPU T4 x2 and run again.') from None
    print('GPUs:', ', '.join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())),
          '| PyTorch', torch.__version__)
    GPUS = min(torch.cuda.device_count(), MAX_GPUS)   # the GPUs the GAN trains on
    if GPUS == 1:
        print('Training on one GPU: each kimg takes about twice as long as on two.')
elif Path('/kaggle').exists():
    raise RuntimeError('No GPU. Choose Settings → Accelerator → GPU T4 x2 and run again.')
else:
    DEVICE, GPUS = torch.device('cpu'), 0
    print('No GPU: running on the CPU (only for a short test run on another computer)')

# Chart style of the project (as in notebooks 01 and 02): light background, dark text, thin grey grid
SURFACE, INK, INK_2, GRID, BLUE, ORANGE = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#2a78d6', '#eb6834'
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'text.color': INK, 'axes.labelcolor': INK_2, 'xtick.color': INK_2, 'ytick.color': INK_2,
    'axes.edgecolor': GRID, 'axes.linewidth': 1, 'grid.color': GRID, 'grid.linewidth': 1,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

## 1. The training images

`splits.csv` comes from notebook 01, and only its **train** rows are used. Each photo goes into the GAN's training
file (a zip file whose `dataset.json` gives every image's class) unchanged when it is a 256 × 256 colour JPEG, which
nearly all PlantVillage photos are; any other photo is cropped to its centre square and resized to 256 × 256. The list
of images is saved as `gan/training_images.csv`: the record that the GAN saw the train split only.

In [ ]:
def find_files(root, name):
    """Every file called `name` under root. Class folders (names with ___) only hold images, so they are skipped."""
    found = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        dirnames[:] = [d for d in dirnames if '___' not in d]
        if name in filenames:
            found.append(Path(dirpath) / name)
    return found

def find_color_dir(root):
    """PlantVillage's `color` folder (the one that holds the Tomato___ class folders) under root."""
    for dirpath, dirnames, _ in os.walk(root):
        dirnames.sort()
        if Path(dirpath).name.lower() == 'color' and any(d.startswith('Tomato') for d in dirnames):
            return Path(dirpath)
        dirnames[:] = [d for d in dirnames if '___' not in d]
    raise FileNotFoundError(f'No PlantVillage "color" folder under {root}. '
                            'Add the Kaggle dataset abdallahalidev/plantvillage-dataset as input.')

def fingerprint(splits):
    return hashlib.sha256('\n'.join(sorted(splits['path'] + ',' + splits['split'])).encode()).hexdigest()[:16]

found = {f: pd.read_csv(f) for f in find_files(INPUT_ROOT, 'splits.csv')}
final_split = [f for f, s in found.items() if fingerprint(s) == SPLIT_FINGERPRINT]
if not final_split:
    raise FileNotFoundError('The final splits.csv of notebook 01 was not found'
                            + (f' (only other versions: {[str(f) for f in found]})' if found else '')
                            + '. Add Input → Your Work → Notebooks → notebook 01, the version that made the '
                              '12,698 / 2,728 / 2,734 split.')
SPLITS_CSV = final_split[0]
splits = found[SPLITS_CSV]
classes = json.loads((SPLITS_CSV.parent / 'classes.json').read_text())
K = len(classes)
assert (splits['label'] == splits['class_folder'].map({c['folder']: c['label'] for c in classes})).all(), \
    'the labels in splits.csv do not match classes.json'
COLOR_DIR = find_color_dir(INPUT_ROOT)
print('Split: ', SPLITS_CSV)
print('Images:', COLOR_DIR)

train = splits[splits['split'] == 'train'].reset_index(drop=True)   # the only rows used in this notebook
train_counts = train['label'].value_counts()
ORDER = [int(i) for i in train_counts.index]   # classes from the most to the fewest training images
CLASS_NAMES = [classes[i]['name'] for i in ORDER]
print(f'\nTrain split: {len(train):,} images (validation and test images are not used here)')
print(pd.DataFrame({'class': CLASS_NAMES, 'train images': [int(train_counts[i]) for i in ORDER]}).to_string(index=False))

In [ ]:
DATA_ZIP = WORK_DIR / f'tomato-train-{RES}.zip'

def zip_entry(rel_path):
    """(name in the zip, file contents, changed?): the photo's own file when it is a RES x RES colour JPEG,
    otherwise its centre square resized to RES x RES (Lanczos), saved as PNG."""
    raw = (COLOR_DIR / rel_path).read_bytes()
    with Image.open(BytesIO(raw)) as img:
        if img.format == 'JPEG' and img.mode == 'RGB' and img.size == (RES, RES):
            return rel_path, raw, False
        img = img.convert('RGB')
        side = min(img.size)
        left, top = (img.width - side) // 2, (img.height - side) // 2
        img = img.crop((left, top, left + side, top + side)).resize((RES, RES), Image.Resampling.LANCZOS)
        out = BytesIO()
        img.save(out, format='PNG')
        return os.path.splitext(rel_path)[0] + '.png', out.getvalue(), True

start = time.time()
with ThreadPoolExecutor(2 * (os.cpu_count() or 2)) as pool:
    entries = list(pool.map(zip_entry, train['path']))
names = [name for name, _, _ in entries]
assert len(set(names)) == len(names), 'two images got the same name in the zip file'
with zipfile.ZipFile(DATA_ZIP, 'w', zipfile.ZIP_STORED) as z:   # JPEGs are compressed already
    for name, data, _ in entries:
        z.writestr(name, data)
    z.writestr('dataset.json', json.dumps({'labels': [[n, int(l)] for n, l in zip(names, train['label'])]}))
changed = sum(c for _, _, c in entries)
del entries

pd.DataFrame({'zip_name': names, 'path': train['path'], 'label': train['label'], 'class': train['class_name']}
             ).to_csv(GAN_OUT / 'training_images.csv', index=False)
# the same images and labels in every session, or a snapshot does not fit
DATA_FINGERPRINT = hashlib.sha256('\n'.join(sorted(f'{n},{l}' for n, l in zip(names, train['label']))).encode()
                                  ).hexdigest()[:16]
print(f'{len(names):,} images in {DATA_ZIP.name} ({DATA_ZIP.stat().st_size / 1e6:,.0f} MB, {time.time() - start:.0f} s); '
      f'{changed} had to be cropped or converted; data fingerprint {DATA_FINGERPRINT}')

## 2. The GAN code

NVIDIA's official PyTorch code for StyleGAN2-ADA is in their `stylegan3` repository: it contains StyleGAN2-ADA as its
`stylegan2` configuration, updated for newer PyTorch than the older `stylegan2-ada-pytorch` repository. It was last
changed in April 2023, so a few lines no longer work today. They are changed here, and each change must apply exactly
once, or the notebook stops:

| File | Change | Why |
|---|---|---|
| `dnnlib/util.py` | its own one-line `strtobool` | `distutils` was removed in Python 3.12 |
| `torch_utils/ops/conv2d_gradfix.py`, `grid_sample_gradfix.py` | no `pkg_resources` version check | `pkg_resources` was removed from setuptools; PyTorch here is 2.x |
| `torch_utils/ops/grid_sample_gradfix.py` | `torch.ops.aten.grid_sampler_2d_backward` | the public way to call this PyTorch function, instead of an internal one that changed between versions. ADA's rotations need it, together with D's R1 regularisation |
| `torch_utils/misc.py` | `super().__init__()` in the image sampler | newer PyTorch no longer accepts the dataset there |
| `training/training_loop.py` | sample grid of 8 images × 10 classes | the original 30 × 16 grid is a 40 MB picture at every snapshot |

On a computer without a GPU (only for a short test), the training loop is also changed to run on the CPU.

StyleGAN also compiles two small CUDA programs, faster versions of two of its operations. If they do not compile on
Kaggle, the same operations in plain PyTorch are used: the same results, slower training.

The training is started by a short script (`train_gan.py`, written in section 4) that sets up what the repository's
`train.py` sets up for `--cfg=stylegan2`, with the settings of the first cell. It adds two things: training stops
cleanly at the time limit, with a last snapshot; and a session starts with the previous session's augmentation
probability *p* instead of starting it again at 0.

In [ ]:
REPO = WORK_DIR / 'stylegan3'

def run(*command):
    subprocess.run([str(c) for c in command], check=True)

def pip_install(*packages):
    run(sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', *packages)

# pytorch-fid: the Inception network for the FID. psutil and click: imported by the GAN code
for module, package in [('pytorch_fid', 'pytorch-fid==0.3.0'), ('psutil', 'psutil'), ('click', 'click')]:
    if importlib.util.find_spec(module) is None:
        pip_install(package)
if GPUS and shutil.which('ninja') is None:   # ninja: needed to compile the CUDA programs
    pip_install('ninja')
    os.environ['PATH'] = str(Path(sys.executable).parent) + os.pathsep + os.environ['PATH']

# (file, original code, new code)
PATCHES = [
    ('dnnlib/util.py',
     'from distutils.util import strtobool',
     "def strtobool(text): return text.strip().lower() in ('y', 'yes', 't', 'true', 'on', '1')  # changed: no distutils"),
    ('torch_utils/ops/conv2d_gradfix.py', 'from pkg_resources import parse_version\n', ''),
    ('torch_utils/ops/conv2d_gradfix.py',
     "_use_pytorch_1_11_api = parse_version(torch.__version__) >= parse_version('1.11.0a') # Allow prerelease builds of 1.11",
     '_use_pytorch_1_11_api = True  # changed: PyTorch 2.x'),
    ('torch_utils/ops/grid_sample_gradfix.py', 'from pkg_resources import parse_version\n', ''),
    ('torch_utils/ops/grid_sample_gradfix.py',
     "_use_pytorch_1_11_api = parse_version(torch.__version__) >= parse_version('1.11.0a') # Allow prerelease builds of 1.11",
     '_use_pytorch_1_11_api = True  # changed: PyTorch 2.x'),
    ('torch_utils/ops/grid_sample_gradfix.py',
     "_use_pytorch_1_12_api = parse_version(torch.__version__) >= parse_version('1.12.0a') # Allow prerelease builds of 1.12",
     '_use_pytorch_1_12_api = True'),
    ('torch_utils/ops/grid_sample_gradfix.py',
     "        op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')\n"
     "        if _use_pytorch_1_12_api:\n"
     "            op = op[0]\n",
     '        op = torch.ops.aten.grid_sampler_2d_backward  # changed: the public way to call it\n'),
    ('torch_utils/misc.py', '        super().__init__(dataset)\n',
     '        super().__init__()  # changed: newer PyTorch takes no argument here\n'),
    ('training/training_loop.py',
     '    gw = np.clip(7680 // training_set.image_shape[2], 7, 32)',
     '    gw = 8  # changed: a smaller sample grid'),
    ('training/training_loop.py',
     '    gh = np.clip(4320 // training_set.image_shape[1], 4, 32)',
     '    gh = max(training_set.label_dim, 4)  # one row per class'),
]
if not GPUS:   # only for a test on a computer without a GPU
    PATCHES += [
        ('training/training_loop.py', "    device = torch.device('cuda', rank)",
         "    device = torch.device('cuda', rank) if torch.cuda.is_available() else torch.device('cpu')"),
        ('training/training_loop.py', '        if rank == 0:\n            phase.start_event',
         "        if rank == 0 and device.type == 'cuda':\n            phase.start_event"),
        ('training/training_loop.py', 'torch.cuda.max_memory_allocated(device) / 2**30', '0.0'),
        ('training/training_loop.py', 'torch.cuda.max_memory_reserved(device) / 2**30', '0.0'),
        ('training/training_loop.py', '        torch.cuda.reset_peak_memory_stats()\n', ''),
        ('training/training_loop.py', '.pin_memory().to(device)', '.to(device)'),
    ]

marker = REPO / 'PATCHED_BY_NOTEBOOK_03'
if not marker.exists():
    if REPO.exists():
        shutil.rmtree(REPO)
    run('git', 'clone', '--quiet', STYLEGAN_REPO, REPO)
    run('git', '-C', REPO, 'checkout', '--quiet', STYLEGAN_COMMIT)
    for rel_path, old, new in PATCHES:
        file = REPO / rel_path
        text = file.read_text(encoding='utf-8')
        if text.count(old) != 1:
            raise RuntimeError(f'{rel_path}: the code to change was found {text.count(old)} times, not once:\n{old}')
        file.write_text(text.replace(old, new, 1), encoding='utf-8')
    marker.write_text(STYLEGAN_COMMIT)
sys.path.insert(0, str(REPO))
print(f'StyleGAN code: commit {STYLEGAN_COMMIT[:7]}, {len(PATCHES)} changes applied')

# Compile the two CUDA programs now (the training processes then load the compiled files) and check that they run
CUDA_KERNELS = False
if GPUS:
    from torch_utils.ops import bias_act, upfirdn2d
    try:
        x = torch.randn(2, 4, 8, 8, device=DEVICE)
        bias_act.bias_act(x, torch.zeros(4, device=DEVICE), act='lrelu', impl='cuda')
        upfirdn2d.upsample2d(x, upfirdn2d.setup_filter([1, 3, 3, 1], device=DEVICE), impl='cuda')
        torch.cuda.synchronize()
        CUDA_KERNELS = bias_act._plugin is not None and upfirdn2d._plugin is not None
    except Exception as e:
        print(f'The CUDA programs did not compile ({type(e).__name__}: {str(e)[-500:]})')
    if not CUDA_KERNELS:   # never try again: use the plain PyTorch versions
        bias_act._init = upfirdn2d._init = lambda: False
    print('CUDA programs:', 'compiled (fast)' if CUDA_KERNELS else 'not used: plain PyTorch, same results, slower')

## 3. Continue from the last session?

The notebook looks among its inputs for `gan/progress.json`, written at the end of every session. If it finds one,
training continues from that session's last snapshot, with its augmentation probability *p* and its kimg count;
otherwise this is session 1. A snapshot only fits the same image size, network width and training images, so those
are checked.

In [ ]:
SETTINGS = {'res': RES, 'channel_base': CHANNEL_BASE, 'mapping_layers': MAPPING_LAYERS, 'data_fingerprint': DATA_FINGERPRINT,
            'batch': BATCH, 'gamma': GAMMA, 'learning_rate': LEARNING_RATE, 'mbstd_group': MBSTD_GROUP,
            'ema_kimg': EMA_KIMG, 'mirror': MIRROR, 'ada_target': ADA_TARGET}
MUST_MATCH = ['res', 'channel_base', 'mapping_layers', 'data_fingerprint']

previous = []
for f in find_files(INPUT_ROOT, 'progress.json'):
    try:
        p = json.loads(f.read_text())
    except (OSError, ValueError):
        continue
    if p.get('kind') == PROGRESS_KIND:
        previous.append(p | {'folder': str(f.parent)})

if previous:
    PREV = max(previous, key=lambda p: (p['kimg'], p['session']))
    PREV_DIR = Path(PREV['folder'])
    wrong = {k: f"{PREV['settings'].get(k)} then, {SETTINGS[k]} now" for k in MUST_MATCH if PREV['settings'].get(k) != SETTINGS[k]}
    if wrong:
        raise RuntimeError(f'The snapshot in {PREV_DIR} does not fit these settings: {wrong}')
    changed = {k: f"{PREV['settings'].get(k)} → {v}" for k, v in SETTINGS.items() if PREV['settings'].get(k) != v}
    SESSION = PREV['session'] + 1
    RESUME_PKL = PREV_DIR / PREV['latest']['file']
    RESUME_KIMG, AUGMENT_P = int(PREV['latest']['kimg']), float(PREV['augment_p'])
    fid_history = pd.read_csv(PREV_DIR / 'fid.csv')
    stats_history = pd.read_csv(PREV_DIR / 'stats.csv')
    print(f'Continuing from {RESUME_PKL}\nSession {SESSION}: starts at {RESUME_KIMG:,} kimg with p = {AUGMENT_P:.3f}. '
          f"Best FID so far {PREV['best']['fid']:.2f} at {PREV['best']['kimg']:,} kimg.")
    if changed:
        print('Settings changed since the last session:', changed)
else:
    PREV, PREV_DIR = None, None
    SESSION, RESUME_PKL, RESUME_KIMG, AUGMENT_P = 1, None, 0, 0.0
    fid_history, stats_history = pd.DataFrame(), pd.DataFrame()
    print('No earlier session among the inputs: session 1 starts from the beginning.\n'
          '(For session 2 and later: Add Input → Your Work → Notebooks → this notebook, its newest version.)')
if RESUME_KIMG >= MAX_KIMG:
    raise RuntimeError(f'Training has reached MAX_KIMG ({MAX_KIMG:,} kimg): nothing to train. Continue with part 2.')

## 4. Train

The training runs in a separate process (one per GPU) and prints a line every 4 kimg, for example:

`tick 12  kimg 1248.0  time 1h 01m  sec/tick 121.0  sec/kimg 30.25  maintenance 0.0  cpumem 3.10  gpumem 5.20  reserved 6.10  augment 0.152`

*sec/kimg* is the speed, and *augment* is ADA's probability *p*. Every 100 kimg it saves a snapshot
(`network-snapshot-<kimg>.pkl`) and a sample grid (`fakes<kimg>.png`, one row per class in the order of
`classes.json`) in `gan/runs/session-NN/`. At the time limit, training finishes the current tick, saves a last
snapshot and stops.

In [ ]:
LAUNCHER_CODE = r'''
"""Starts StyleGAN2-ADA training with the settings in the JSON file named by the GAN_CONFIG environment variable.
Written by notebook 03_gan_train. It sets up what the repository's train.py sets up for --cfg=stylegan2, and adds a
deadline (training stops cleanly, with a last snapshot) and the starting augmentation probability of a session."""
import json, os, sys, tempfile, time
import torch

CFG = json.load(open(os.environ["GAN_CONFIG"]))
sys.path.insert(0, CFG["repo"])
import dnnlib
from training import training_loop
from torch_utils import custom_ops, training_stats
from torch_utils.ops import bias_act, upfirdn2d

if not CFG["cuda_kernels"]:   # they did not compile: use the plain PyTorch versions (same results, slower)
    bias_act._init = upfirdn2d._init = lambda: False


def stop_now():
    """True once the deadline has passed. Every GPU must give the same answer, so GPU 0 decides for all."""
    stop = time.time() > CFG["deadline"]
    if torch.distributed.is_available() and torch.distributed.is_initialized():
        flag = torch.tensor([float(stop)], device=torch.device("cuda", torch.distributed.get_rank()))
        torch.distributed.broadcast(flag, src=0)
        stop = bool(flag.item())
    return stop


def options():
    E = dnnlib.EasyDict
    width = dict(channel_base=CFG["channel_base"], channel_max=512)
    adam = dict(class_name="torch.optim.Adam", betas=[0, 0.99], eps=1e-8, lr=CFG["lr"])
    return E(
        run_dir=CFG["run_dir"],
        training_set_kwargs=E(class_name="training.dataset.ImageFolderDataset", path=CFG["data"], use_labels=True,
                              max_size=CFG["num_images"], xflip=CFG["mirror"], resolution=CFG["res"],
                              random_seed=CFG["seed"]),
        data_loader_kwargs=E(pin_memory=CFG["pin_memory"], prefetch_factor=2, num_workers=CFG["workers"]),
        G_kwargs=E(class_name="training.networks_stylegan2.Generator", z_dim=512, w_dim=512,
                   mapping_kwargs=E(num_layers=CFG["mapping_layers"]), fused_modconv_default="inference_only", **width),
        D_kwargs=E(class_name="training.networks_stylegan2.Discriminator", block_kwargs=E(freeze_layers=0),
                   mapping_kwargs=E(), epilogue_kwargs=E(mbstd_group_size=CFG["mbstd_group"]), **width),
        G_opt_kwargs=E(adam), D_opt_kwargs=E(adam),
        loss_kwargs=E(class_name="training.loss.StyleGAN2Loss", r1_gamma=CFG["gamma"], style_mixing_prob=0.9,
                      pl_weight=2, pl_no_weight_grad=True),
        augment_kwargs=E(class_name="training.augment.AugmentPipe", xflip=1, rotate90=1, xint=1, scale=1, rotate=1,
                         aniso=1, xfrac=1, brightness=1, contrast=1, lumaflip=1, hue=1, saturation=1),
        ada_target=CFG["ada_target"], augment_p=CFG["augment_p"], G_reg_interval=4, metrics=[],
        num_gpus=CFG["gpus"], batch_size=CFG["batch"], batch_gpu=CFG["batch_gpu"], ema_kimg=CFG["ema_kimg"],
        total_kimg=CFG["max_kimg"], kimg_per_tick=CFG["tick_kimg"],
        image_snapshot_ticks=CFG["snapshot_ticks"], network_snapshot_ticks=CFG["snapshot_ticks"],
        random_seed=CFG["seed"], resume_pkl=CFG["resume_pkl"], resume_kimg=CFG["resume_kimg"], abort_fn=stop_now,
    )


def subprocess_fn(rank, c, temp_dir):
    """One process per GPU, as in train.py."""
    dnnlib.util.Logger(file_name=os.path.join(c.run_dir, "log.txt"), file_mode="a", should_flush=True)
    if c.num_gpus > 1:
        init_file = os.path.abspath(os.path.join(temp_dir, ".torch_distributed_init"))
        torch.distributed.init_process_group(backend="nccl", init_method=f"file://{init_file}", rank=rank,
                                             world_size=c.num_gpus)
    training_stats.init_multiprocessing(rank=rank, sync_device=torch.device("cuda", rank) if c.num_gpus > 1 else None)
    if rank != 0:
        custom_ops.verbosity = "none"
    training_loop.training_loop(rank=rank, **c)


if __name__ == "__main__":
    c = options()
    os.makedirs(c.run_dir)
    with open(os.path.join(c.run_dir, "training_options.json"), "wt") as f:
        json.dump(c, f, indent=2, default=str)
    with tempfile.TemporaryDirectory() as temp_dir:
        if c.num_gpus == 1:
            subprocess_fn(0, c, temp_dir)
        else:
            torch.multiprocessing.spawn(fn=subprocess_fn, args=(c, temp_dir), nprocs=c.num_gpus)
'''
LAUNCHER = WORK_DIR / 'train_gan.py'
LAUNCHER.write_text(LAUNCHER_CODE, encoding='utf-8')

RUN_DIR = GAN_OUT / 'runs' / f'session-{SESSION:02d}'
if RUN_DIR.exists():   # only when the notebook is run twice in the same Kaggle session
    shutil.rmtree(RUN_DIR)
DEADLINE = NOTEBOOK_START + TRAIN_HOURS * 3600
CONFIG = {
    'repo': str(REPO), 'data': str(DATA_ZIP), 'num_images': len(names), 'res': RES, 'mirror': MIRROR,
    'gpus': max(GPUS, 1), 'batch': BATCH, 'batch_gpu': BATCH_PER_GPU, 'workers': 2, 'pin_memory': GPUS > 0,
    'channel_base': CHANNEL_BASE, 'mapping_layers': MAPPING_LAYERS, 'mbstd_group': MBSTD_GROUP, 'gamma': GAMMA,
    'lr': LEARNING_RATE, 'ema_kimg': EMA_KIMG, 'ada_target': ADA_TARGET, 'augment_p': AUGMENT_P,
    'max_kimg': MAX_KIMG, 'tick_kimg': TICK_KIMG, 'snapshot_ticks': SNAPSHOT_KIMG // TICK_KIMG,
    'seed': SESSION - 1,   # a new order of the training images in every session
    'resume_pkl': str(RESUME_PKL) if RESUME_PKL else None, 'resume_kimg': RESUME_KIMG,
    'deadline': DEADLINE, 'cuda_kernels': CUDA_KERNELS, 'run_dir': str(RUN_DIR),
}
CONFIG_JSON = WORK_DIR / 'train_gan.json'
CONFIG_JSON.write_text(json.dumps(CONFIG, indent=1))

def kimg_of(path):
    return int(re.search(r'(\d+)\.pkl$', Path(path).name).group(1))

def train_gan():
    """Run train_gan.py, show its output here, and return its exit code."""
    env = dict(os.environ, GAN_CONFIG=str(CONFIG_JSON), PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
    proc = subprocess.Popen([sys.executable, str(LAUNCHER)], cwd=REPO, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
                            start_new_session=True)

    def kill():   # only if training hangs: normally it stops by itself at the deadline
        if proc.poll() is None:
            print('\nTraining did not stop by itself: stopping it now.', flush=True)
            if hasattr(os, 'killpg'):
                os.killpg(proc.pid, signal.SIGKILL)
            else:
                proc.kill()

    timer = threading.Timer(max(DEADLINE + 45 * 60 - time.time(), 60), kill)
    timer.daemon = True
    timer.start()
    for line in proc.stdout:
        print(line, end='', flush=True)
    timer.cancel()
    return proc.wait()

print(f'Session {SESSION}: training on {max(GPUS, 1)} {"GPU" if GPUS else "CPU"}(s) until '
      f'{datetime.fromtimestamp(DEADLINE, timezone.utc):%H:%M} UTC ({TRAIN_HOURS} h after the notebook started)\n')
exit_code = train_gan()
new_snapshots = sorted((p for p in RUN_DIR.glob('network-snapshot-*.pkl') if kimg_of(p) > RESUME_KIMG), key=kimg_of)
if not new_snapshots:
    raise RuntimeError(f'Training stopped (exit code {exit_code}) before its first new snapshot. '
                       + ('The error is in the lines above.' if exit_code else 'Is TRAIN_HOURS long enough?'))
if exit_code != 0:
    print(f'\nWARNING: training stopped with an error (exit code {exit_code}); '
          f'the {len(new_snapshots)} snapshot(s) saved before it are used.')
print(f'\nThis session: {len(new_snapshots)} new snapshot(s), up to {kimg_of(new_snapshots[-1]):,} kimg')

## 5. FID of each snapshot

The **FID** (Fréchet Inception Distance) compares generated and real images through the features an Inception network
(trained on ImageNet) sees in them: it is the distance between the mean and spread of the features of 10,000
generated images and of all 12,698 training images. 0 means the same statistics; lower is better. The generated
images have the same class shares as the train split, and the same random numbers are used for every snapshot, so
the snapshots are compared fairly. The Inception network is the one of `pytorch-fid`, the usual PyTorch version of
the original FID network. As usual for GANs, the FID is measured against the GAN's own training images; validation
and test images are not involved.

The FID here compares snapshots of this GAN. It is not comparable with FIDs in papers about other datasets.
Part 2 also measures it per class.

In [ ]:
from pytorch_fid.inception import InceptionV3

inception = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(DEVICE).eval()
FID_BATCH = 50

@torch.inference_mode()
def inception_features(images):
    """2048 features per image; images: (n, 3, H, W) 8-bit pixels."""
    return inception(images.to(DEVICE).float() / 255)[0].flatten(1).double().cpu()

def mean_cov(features):
    f = features.numpy()
    return f.mean(0), np.cov(f, rowvar=False)

def frechet_distance(a, b):
    (m1, s1), (m2, s2) = a, b
    sqrt_trace = np.sqrt(np.linalg.eigvals(s1 @ s2).astype(complex)).real.sum()   # trace of the square root of s1 s2
    return float(np.sum((m1 - m2) ** 2) + np.trace(s1) + np.trace(s2) - 2 * sqrt_trace)

def zip_image(z, name):
    with Image.open(BytesIO(z.read(name))) as img:
        return np.asarray(img.convert('RGB'))

def load_generator(pkl):
    """G_ema of a snapshot: the generator used for images."""
    with open(pkl, 'rb') as f:
        return pickle.load(f)['G_ema'].to(DEVICE).eval().requires_grad_(False)

@torch.inference_mode()
def generate(G, z, labels):
    """Images of the given classes as (n, 3, RES, RES) 8-bit pixels."""
    out = []
    for i in range(0, len(z), FID_BATCH):
        c = torch.nn.functional.one_hot(labels[i:i + FID_BATCH], G.c_dim).float().to(DEVICE)
        img = G(z[i:i + FID_BATCH].to(DEVICE), c, truncation_psi=1, noise_mode='const')
        out.append((img * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu())
    return torch.cat(out)

start = time.time()
real_features = []
with zipfile.ZipFile(DATA_ZIP) as z:
    for i in range(0, len(names), FID_BATCH):
        batch = np.stack([zip_image(z, n) for n in names[i:i + FID_BATCH]])
        real_features.append(inception_features(torch.from_numpy(batch).permute(0, 3, 1, 2)))
REAL_STATS = mean_cov(torch.cat(real_features))
print(f'Features of the {len(names):,} training images: {time.time() - start:.0f} s')

# FID_IMAGES labels in the class shares of the train split, and fixed random numbers
share = train_counts.sort_index().to_numpy() / len(train)
counts = np.floor(FID_IMAGES * share).astype(int)
counts[np.argmax(counts)] += FID_IMAGES - counts.sum()
FID_LABELS = torch.from_numpy(np.repeat(np.arange(K), counts))
FID_Z = torch.from_numpy(np.random.RandomState(SAMPLE_SEED).randn(FID_IMAGES, 512)).float()

new_fid = []
for pkl in new_snapshots:
    start = time.time()
    G = load_generator(pkl)
    features = [inception_features(generate(G, FID_Z[i:i + 500], FID_LABELS[i:i + 500])) for i in range(0, FID_IMAGES, 500)]
    del G
    new_fid.append({'session': SESSION, 'kimg': kimg_of(pkl), 'fid': frechet_distance(mean_cov(torch.cat(features)), REAL_STATS),
                    'snapshot': pkl.name})
    print(f'{pkl.name}: FID {new_fid[-1]["fid"]:7.2f}   ({time.time() - start:.0f} s)')
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

## 6. Keep the snapshots and save the progress

Two snapshots are kept in `gan/snapshots/`: the **latest** (the next session continues from it) and the **best** so far
(lowest FID, from any session; part 2 uses it). The other snapshots of this session are deleted, to keep the output
small. `gan/progress.json` tells the next session where to continue; `gan/fid.csv` and `gan/stats.csv` hold the
history of all sessions.

In [ ]:
STATS_COLUMNS = {'Progress/kimg': 'kimg', 'Progress/augment': 'augment_p', 'Timing/sec_per_kimg': 'sec_per_kimg',
                 'Loss/signs/real': 'd_sign_real', 'Loss/scores/real': 'd_score_real', 'Loss/scores/fake': 'd_score_fake',
                 'Loss/G/loss': 'g_loss', 'Loss/D/loss': 'd_loss', 'Loss/r1_penalty': 'r1_penalty',
                 'Resources/peak_gpu_mem_gb': 'gpu_mem_gb', 'Timing/total_hours': 'hours'}
stats_rows = []
for line in (RUN_DIR / 'stats.jsonl').read_text().splitlines():
    tick = json.loads(line)
    stats_rows.append({'session': SESSION, **{new: tick[old]['mean'] if old in tick else np.nan
                                              for old, new in STATS_COLUMNS.items()}})
stats_new = pd.DataFrame(stats_rows)
stats_all = pd.concat([stats_history, stats_new], ignore_index=True)
stats_all.to_csv(GAN_OUT / 'stats.csv', index=False)

fid_new = pd.DataFrame(new_fid)
fid_new['augment_p'] = [stats_new.loc[(stats_new['kimg'] - k).abs().idxmin(), 'augment_p'] for k in fid_new['kimg']]
fid_all = pd.concat([fid_history, fid_new], ignore_index=True)
fid_all.to_csv(GAN_OUT / 'fid.csv', index=False)
fid_all.to_csv(RESULTS_OUT / '03_fid.csv', index=False)

latest = fid_new.iloc[-1]
best = fid_all.loc[fid_all['fid'].idxmin()]
SNAP_DIR = GAN_OUT / 'snapshots'
SNAP_DIR.mkdir(exist_ok=True)
shutil.move(RUN_DIR / latest['snapshot'], SNAP_DIR / latest['snapshot'])
if best['snapshot'] != latest['snapshot']:
    if best['session'] == SESSION:
        shutil.move(RUN_DIR / best['snapshot'], SNAP_DIR / best['snapshot'])
    else:   # the best one is from an earlier session: copy it from the input
        shutil.copy(PREV_DIR / PREV['best']['file'], SNAP_DIR / best['snapshot'])
for pkl in RUN_DIR.glob('network-snapshot-*.pkl'):
    pkl.unlink()

train_seconds = stats_new['hours'].iloc[-1] * 3600
speed = float(stats_new['sec_per_kimg'].iloc[1:].median()) if len(stats_new) > 1 else float('nan')
PROGRESS = {
    'kind': PROGRESS_KIND, 'session': SESSION, 'kimg': int(latest['kimg']), 'augment_p': float(stats_new['augment_p'].iloc[-1]),
    'latest': {'file': f"snapshots/{latest['snapshot']}", 'kimg': int(latest['kimg']), 'fid': float(latest['fid'])},
    'best': {'file': f"snapshots/{best['snapshot']}", 'kimg': int(best['kimg']), 'fid': float(best['fid']),
             'session': int(best['session'])},
    'settings': SETTINGS, 'max_kimg': MAX_KIMG, 'fid_images': FID_IMAGES, 'sample_seed': SAMPLE_SEED,
    'training_images': 'train split only', 'split_csv': str(SPLITS_CSV), 'split_fingerprint': SPLIT_FINGERPRINT,
    'stylegan_code': f'{STYLEGAN_REPO} @ {STYLEGAN_COMMIT}', 'cuda_kernels': CUDA_KERNELS,
    'gpus': GPUS, 'device': torch.cuda.get_device_name() if GPUS else 'CPU', 'torch': torch.__version__,
    'sec_per_kimg': round(speed, 2), 'train_hours_total': round((PREV['train_hours_total'] if PREV else 0) + train_seconds / 3600, 2),
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
}
(GAN_OUT / 'progress.json').write_text(json.dumps(PROGRESS, indent=1))
shutil.copy(GAN_OUT / 'progress.json', RESULTS_OUT / '03_progress.json')
print(f"Latest: {latest['snapshot']} (FID {latest['fid']:.2f}).  Best: {best['snapshot']} (FID {best['fid']:.2f}, "
      f"session {int(best['session'])}).")
print(f"This session: {stats_new['kimg'].iloc[-1] - RESUME_KIMG:,.0f} kimg in {train_seconds / 3600:.1f} h, "
      f"{speed:.1f} s per kimg; all sessions: {PROGRESS['train_hours_total']} h of training.")

## 7. Charts and samples

**Progress chart.** Left: the FID of every snapshot, all sessions (dashed lines: the start of a session). Middle:
ADA's probability *p*. Right: ADA's overfitting measure, the average sign of D's output on the real training images
(+1: D calls all of them real; 0: half). A D that memorises the training images pushes it up; ADA then raises *p*
until the measure is back near 0.6.

**Samples.** For each class, 2 real training photos and 8 generated images from the latest snapshot. The 8 generated
images of a row use the same 8 random numbers in every row and every session, so the rows differ only by the class,
and the same picture of a later session shows the progress.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
ax = axes[0]
ax.plot(fid_all['kimg'], fid_all['fid'], color=BLUE, marker='o', markersize=4, linewidth=1.5)
ax.plot(best['kimg'], best['fid'], 'o', color=ORANGE, markersize=9, label=f"best: {best['fid']:.1f} at {int(best['kimg']):,} kimg")
ax.set_yscale('log')
ax.set_title('FID (lower is better)')
ax.legend(frameon=False)
ax = axes[1]
for session, part in stats_all.groupby('session'):
    ax.plot(part['kimg'], part['augment_p'], color=BLUE, linewidth=1.2)
ax.set_title('ADA augmentation probability p')
ax = axes[2]
ax.plot(stats_all['kimg'], stats_all['d_sign_real'], color=BLUE, linewidth=0.8, alpha=0.8)
ax.axhline(ADA_TARGET, color=ORANGE, linewidth=1.2, label=f'ADA target {ADA_TARGET}')
ax.set_title('D on real images (ADA overfitting measure)')
ax.legend(frameon=False)
for ax in axes:
    ax.set_xlabel('kimg')
    ax.grid(True, axis='y')
    for start_kimg in stats_all.groupby('session')['kimg'].min().iloc[1:]:
        ax.axvline(start_kimg, color=INK_2, linewidth=0.8, linestyle='--')
fig.tight_layout()
fig.savefig(RESULTS_OUT / '03_progress.png')
plt.show()

In [ ]:
REALS, FAKES = 2, 8
rng = np.random.RandomState(SAMPLE_SEED)
sample_z = torch.from_numpy(rng.randn(FAKES, 512)).float()   # the same 8 for every class and every session
names_of = {label: [n for n, l in zip(names, train['label']) if l == label] for label in range(K)}
G = load_generator(SNAP_DIR / latest['snapshot'])
fig, axes = plt.subplots(K, REALS + FAKES, figsize=(1.35 * (REALS + FAKES) + 1.6, 1.35 * K),
                         gridspec_kw={'wspace': 0.04, 'hspace': 0.04})
with zipfile.ZipFile(DATA_ZIP) as z:
    for row, label in enumerate(ORDER):
        reals = [zip_image(z, n) for n in rng.choice(names_of[label], REALS, replace=False)]
        fakes = generate(G, sample_z, torch.full((FAKES,), label)).permute(0, 2, 3, 1).numpy()
        for col, img in enumerate(reals + list(fakes)):
            ax = axes[row, col]
            ax.imshow(img)
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(col < REALS)
                spine.set_color(ORANGE)
                spine.set_linewidth(2)
        axes[row, 0].set_ylabel(CLASS_NAMES[row], rotation=0, ha='right', va='center', fontsize=10, color=INK)
del G
axes[0, 0].set_title('real', fontsize=10, loc='center', color=ORANGE)
axes[0, REALS].set_title('generated →', fontsize=10, loc='left')
fig.subplots_adjust(top=0.95)
fig.suptitle(f"Session {SESSION}, {int(latest['kimg']):,} kimg, FID {latest['fid']:.1f}: per class 2 real training "
             'photos (orange frame) and 8 generated images', x=0.12, y=0.99, ha='left', fontsize=12, fontweight='bold')
SAMPLES_PNG = RESULTS_OUT / f"03_samples_kimg{int(latest['kimg']):05d}.png"
fig.savefig(SAMPLES_PNG)
plt.show()

if PREV_DIR is not None:   # keep the sample pictures of the earlier sessions in this output too
    for f in (PREV_DIR.parent / 'results').glob('03_samples_kimg*.png'):
        if not (RESULTS_OUT / f.name).exists():
            shutil.copy(f, RESULTS_OUT / f.name)
print('Output files:')
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file() and f.parent.name in ('gan', 'snapshots', 'results'):
        print(f'  {f.relative_to(OUT_DIR)}  ({f.stat().st_size / 1e6:,.1f} MB)')

## Checklist

1. **The training lines** (section 4): `sec/kimg` gives the speed; one session gives at most about TRAIN_HOURS × 3600 / sec/kimg
   kimg. `augment` (*p*) starts at 0 and rises as D starts to memorise; values up to about 0.5 are normal for a
   dataset of this size. If *p* keeps climbing towards 0.8 or more, D memorises hard, and generated images may start
   to show augmentation effects (odd colours, rotated edges).
2. **FID** (left chart): it falls quickly at first, then slowly. Stop when it has not improved for about 500 kimg (5
   snapshots), or when the time box is used up. A FID that rises for several snapshots in a row means training is
   getting worse; the best snapshot is kept anyway.
3. **Samples**: leaves look like leaves on PlantVillage's grey background; each row shows the symptoms of its class
   (for example yellow curled leaves for Yellow leaf curl virus, a mottled pattern for Mosaic virus, a plain green
   leaf for Healthy); the rows differ from each other. Note classes whose images look wrong or alike: part 2 checks
   every generated image with the ResNet50 classifier and drops the ones that do not show their class.
4. **Copies**: generated images should not be copies of training photos. Part 2 checks this by finding the nearest
   training photo of each generated image.

**Next session:** Add Input → Your Work → Notebooks → this notebook (newest version), then commit again.
**When training is finished:** part 2 (`03_gan_generate`) uses the best snapshot, `gan/snapshots/` of the last
session's output.